In [ ]:
# print("123")

In [1]:
!pip install minsearch


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
!pip install pydantic


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
!pip install python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [4]:
!pip install google-genai


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [5]:
!pip install tqdm


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [6]:
import os
import json
import re
import pandas as pd
from ingest import load_faq_data
from evaluation_utils import llm_structured, llm_structured_retry, map_progress
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from pydantic import BaseModel
from dotenv import load_dotenv
load_dotenv()

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [7]:
from google import genai
from google.genai import types
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [8]:
model='gemini-3.5-flash'

In [9]:
documents = load_faq_data()
# documents[10]

In [10]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

# len(documents_llm)

In [11]:
documents = documents_llm

In [12]:
doc = documents[0]

In [13]:
class Question(BaseModel):
    questions: list[str]

In [14]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [ ]:
# user_prompt = json.dumps(doc)

In [49]:
# print(user_prompt)

In [15]:
user_prompt = "FAQ Record: The course platform is accessible 24/7, and certificates are issued automatically upon passing the final exam."

In [16]:
messages = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(text=user_prompt), 
            types.Part.from_text(text=data_gen_instructions)
        ]
    )
]

config = types.GenerateContentConfig(
    temperature=0.2,
    max_output_tokens=2000,
    response_mime_type="application/json",  # Forces Gemini to output strict JSON
    response_schema=Question                # Enforces your structural blueprint
)

response = client.models.generate_content(
    model=model,
    contents=messages,
    config=config
)

In [17]:
raw_text = response.text.strip()

In [18]:
try:
    if response.parsed is not None:
        questions_list = response.parsed.questions
    else:
        questions_list = json.loads(raw_text).get("questions", [])
except json.JSONDecodeError:
    # If it is still truncated or malformed, attempt to clean/patch it manually
    print("Received malformed JSON string from API. Attempting repair...")
    
    # 1. Strip markdown code fence wrappers if Gemini accidentally included them
    clean_text = re.sub(r"^```json\s*|\s*```$", "", raw_text, flags=re.MULTILINE).strip()
    
    # 2. Basic auto-closure check if it got cut off near the end
    if not clean_text.endswith("]}"):
        if clean_text.endswith('"'): clean_text += "]}"
        elif not clean_text.endswith(']'): clean_text += '"]}'
    
    try:
        questions_list = json.loads(clean_text).get("questions", [])
    except Exception as e:
        # Final safety net fallback
        print(f"Could not repair JSON. Raw response was: \n{raw_text}")
        questions_list = []

In [20]:
print("Final Output Questions:", questions_list)

Final Output Questions: ['Can I access the course platform at any time?', 'Is there a specific schedule I need to follow to access the course content?', 'How do I receive my certificate once I finish the course?', 'Will my certificate be sent to me automatically, or do I need to request it?', 'When exactly will I get my certificate after completing the final assessment?']


In [21]:
doc

{'id': 'ab183bd688',
 'course': 'machine-learning-zoomcamp',
 'section': 'Miscellaneous',
 'question': "My homework answer doesn't match any of the options",
 'answer': "Common causes, in order of frequency:\n\n1. Wrong column slice or filter — apply filters BEFORE selecting columns / `.head(n)` / `.values`.\n2. Log transform applied where it shouldn't be (or not applied where it should).\n3. Rounding too early — only round the final answer, not intermediate values, unless explicitly told to.\n4. Different sklearn / numpy / Python versions — pin them via `requirements.txt`, `Pipfile.lock`, or `uv.lock`.\n5. Different train/val/test split logic — `train_test_split` shuffles by default; manual `np.random.shuffle` produces a different ordering than sklearn's.\n\nIf after these checks your answer still doesn't match, pick the closest option — the homework explicitly allows it."}

In [19]:
result, usage = llm_structured(
    client,
    data_gen_instructions,
    user_prompt,
    Question
)

# print(result.questions)

In [ ]:
# # Extract token usage from metadata
# usage = response.usage_metadata
# prompt_tokens = usage.prompt_token_count
# candidate_tokens = usage.candidates_token_count

# # Print individual metrics
# print(f"Prompt (Input) Tokens: {prompt_tokens}")
# print(f"Candidates (Output) Tokens: {candidate_tokens}")
# print(f"Total Tokens Used: {usage.total_token_count}")

Prompt (Input) Tokens: 115
Candidates (Output) Tokens: 83
Total Tokens Used: 1426


In [20]:
usage = response.usage_metadata

In [21]:
def calc_price(usage):
    usage = response.usage_metadata
    prompt_tokens = usage.prompt_token_count
    candidate_tokens = usage.candidates_token_count
    
    input_price_per_million = 0.75
    output_price_per_million = 4.50

    input_cost = (prompt_tokens / 1_000_000) * input_price_per_million
    output_cost = (candidate_tokens / 1_000_000) * output_price_per_million
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

In [22]:
def calc_total_price(usages):
    total_cost = 0.0

    for usage in usages:
        cost = calc_price(usage)
        total_cost = total_cost + cost["total_cost"]

    return total_cost

In [23]:
calc_price(usage)

{'input_cost': 8.625000000000001e-05,
 'output_cost': 0.000459,
 'total_cost': 0.0005452499999999999}

In [24]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is the portal open for studying at like 3 AM, or does it close at night?',
  'document': '74eb249bbf'},
 {'question': 'Can I log in to do the coursework whenever I want, or are there specific hours?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to email someone to get my credential after I clear the last test?',
  'document': '74eb249bbf'},
 {'question': 'How long do I have to wait to get my certificate after I finish the last test?',
  'document': '74eb249bbf'},
 {'question': 'Does the system generate the certificate on its own once I get a passing grade on the test?',
  'document': '74eb249bbf'}]

In [25]:
pd.DataFrame(records)

,question,document
0,"Is the portal open for studying at like 3 AM, ...",74eb249bbf
1,Can I log in to do the coursework whenever I w...,74eb249bbf
2,Do I need to email someone to get my credentia...,74eb249bbf
3,How long do I have to wait to get my certifica...,74eb249bbf
4,Does the system generate the certificate on it...,74eb249bbf


In [26]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        client,
        data_gen_instructions,
        user_prompt,
        Question
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [29]:
generate_ground_truth(doc)

([{'question': 'Where can I get the Zoom link for the upcoming live Q&A session?',
   'document': '489dd1c9d9'},
  {'question': 'How do we access the live workshops? Will the links be shared on Slack or Telegram?',
   'document': '489dd1c9d9'},
  {'question': "What's the best way to submit a question during office hours so it doesn't get lost?",
   'document': '489dd1c9d9'},
  {'question': 'Is there a specific platform we use to watch the live streams, or is it on YouTube?',
   'document': '489dd1c9d9'},
  {'question': 'Can I just ask my questions in the live chat during the broadcast?',
   'document': '489dd1c9d9'}],
 GenerateContentResponseUsageMetadata(
   candidates_token_count=97,
   prompt_token_count=292,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=292
     ),
   ],
   thoughts_token_count=978,
   total_token_count=1367
 ))

In [30]:
ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)


100%|██████████| 5/5 [00:44<00:00,  8.83s/it]


In [ ]:
with ThreadPoolExecutor(max_workers=2) as pool:
    results = map_progress(pool, documents, generate_ground_truth)    

 20%|██        | 23/113 [00:49<03:15,  2.17s/it]


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.5-flash\nPlease retry in 40.340687874s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.5-flash'

In [28]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)
    
len(ground_truth)

NameError: name 'results' is not defined

In [ ]:
ground_truth[0]

In [ ]:
total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

In [ ]:
df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [ ]:
len(df_ground_truth)